# 02 — Building Expectations: Olist Data Audit

**Цель ноутбука:** превратить каталог из дня 1 (16 находок) в работающие Expectation Suites через Great Expectations 1.x, прогнать валидацию и сгенерировать HTML-отчёт (Data Docs).

**Что такое Great Expectations за 30 секунд.**
GE — это библиотека для тестирования качества данных. Аналогия: pytest тестирует код, GE тестирует данные. Базовые сущности:
- **Expectation** — одна проверка («колонка `price` всегда положительна»)
- **Expectation Suite** — набор проверок для одной таблицы
- **Validation Definition** — связка «эту suite применить к этим данным»
- **Checkpoint** — оркестратор: запускает validation и пишет результаты в отчёт
- **Data Docs** — автогенерируемый HTML-отчёт с результатами

**План работы:**
1. Setup: инициализация контекста (создаст папку `gx/`)
2. Регистрация датасета (Olist CSV-файлы)
3. Полная suite для `orders` — пример «как делать» с разбором
4. TODO-секции: 4 suite для других таблиц (вы пишете сами по шаблону)
5. Запуск всех проверок через Checkpoint
6. Генерация Data Docs

**К концу дня 2** у вас должно быть: 5 Expectation Suites, прогнанная валидация, готовый HTML-отчёт.

## 0. Setup

In [1]:
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd
from pathlib import Path

print(f"GE version: {gx.__version__}")
print(f"Working directory: {Path.cwd()}")

GE version: 1.17.2
Working directory: C:\Users\Лика\Desktop\data-audit-pipeline\notebooks


## 1. Инициализируем GE Context

Context — это «корневой объект» в GE. В режиме `file` он создаст рядом папку `gx/`, где будут храниться все определения (datasources, suites, checkpoints) в YAML-файлах. Это значит, что определения версионируются в Git и переживают перезапуск ядра.

**Важно:** запускаем из корня проекта (там, где `requirements.txt`), а не из `notebooks/`. Если ноутбук запущен из `notebooks/`, переключитесь:

In [2]:
# Корень проекта (на уровень выше папки notebooks)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print(f"Project root: {PROJECT_ROOT}")

import os
os.chdir(PROJECT_ROOT)

context = gx.get_context(mode="file", project_root_dir=".")
print(f"\nContext type: {type(context).__name__}")
print(f"GX directory: {PROJECT_ROOT / 'gx'}")

Project root: C:\Users\Лика\Desktop\data-audit-pipeline

Context type: FileDataContext
GX directory: C:\Users\Лика\Desktop\data-audit-pipeline\gx


После выполнения этой ячейки в корне проекта появится папка `gx/`. Не пугайтесь, это нормально — там лежат YAML-файлы с конфигурацией.

## 2. Регистрируем данные

Подключаем папку с CSV-файлами как Data Source. Дальше каждый CSV регистрируем как Asset с одним Batch Definition (= «весь файл целиком как один батч»).

In [3]:
# Идемпотентно: пересоздаём data source, если уже был
DATASOURCE_NAME = "olist"

if DATASOURCE_NAME in [ds.name for ds in context.data_sources.all()]:
    context.data_sources.delete(DATASOURCE_NAME)

data_source = context.data_sources.add_pandas_filesystem(
    name=DATASOURCE_NAME,
    base_directory="./data/raw"
)
print(f"Data source created: {data_source.name}")

Data source created: olist


In [4]:
# Регистрируем все 9 таблиц как assets
TABLE_FILES = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

batch_definitions = {}
for asset_name, filename in TABLE_FILES.items():
    asset = data_source.add_csv_asset(name=asset_name)
    batch_def = asset.add_batch_definition_path(
        name=f"{asset_name}_full",
        path=filename
    )
    batch_definitions[asset_name] = batch_def
    print(f"  registered: {asset_name:25} -> {filename}")

  registered: customers                 -> olist_customers_dataset.csv
  registered: orders                    -> olist_orders_dataset.csv
  registered: order_items               -> olist_order_items_dataset.csv
  registered: order_payments            -> olist_order_payments_dataset.csv
  registered: order_reviews             -> olist_order_reviews_dataset.csv
  registered: products                  -> olist_products_dataset.csv
  registered: sellers                   -> olist_sellers_dataset.csv
  registered: geolocation               -> olist_geolocation_dataset.csv
  registered: category_translation      -> product_category_name_translation.csv


## 3. Пример suite целиком: `orders`

Это **референсный пример**. Разберитесь с ним детально — дальше по шаблону сделаете 4 другие suite самостоятельно.

Из вашего каталога к `orders` относятся находки: #1 (даты), #4 (delivered_date nulls), #11 (delivered < carrier), #12 (carrier < approved), #13 (lead_time > 0), #14 (FK customer_id), #15 (empty orders).

Заметка: некоторые из них (FK между таблицами, multi-column date order) выражаются нетривиально. Делаем то, что прямо ложится на GE, остальное — в день 3 как custom checks.

In [5]:
# Идемпотентно: пересоздаём suite, если уже существовала
SUITE_NAME_ORDERS = "orders_suite"

if SUITE_NAME_ORDERS in [s.name for s in context.suites.all()]:
    context.suites.delete(SUITE_NAME_ORDERS)

orders_suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME_ORDERS))
print(f"Suite created: {orders_suite.name}")

Suite created: orders_suite


### 3.1 Колонки и типы

In [6]:
# Ожидаемый набор колонок в таблице
orders_suite.add_expectation(
    gxe.ExpectTableColumnsToMatchSet(
        column_set=[
            'order_id', 'customer_id', 'order_status',
            'order_purchase_timestamp', 'order_approved_at',
            'order_delivered_carrier_date', 'order_delivered_customer_date',
            'order_estimated_delivery_date',
        ],
        exact_match=True,
    )
)

# order_id — уникален и не null
orders_suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='order_id'))
orders_suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column='order_id'))

# customer_id — не null (FK-проверку добавим в день 3 как custom check)
orders_suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='customer_id'))

ExpectColumnValuesToNotBeNull(id='ae2ab79b-5da7-4c80-ac38-8d6880847069', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='customer_id', mostly=1, row_condition=None, condition_parser=None)

### 3.2 order_status — закрытый набор значений

In [7]:
# Из find #7 в day 1 вы видели 8 уникальных статусов
ORDER_STATUSES = [
    'delivered', 'shipped', 'canceled', 'unavailable',
    'invoiced', 'processing', 'created', 'approved'
]

orders_suite.add_expectation(
    gxe.ExpectColumnValuesToBeInSet(
        column='order_status',
        value_set=ORDER_STATUSES,
    )
)

ExpectColumnValuesToBeInSet(id='7270f6b3-d2db-481b-9404-2999241b828f', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_status', mostly=1, row_condition=None, condition_parser=None, value_set=['delivered', 'shipped', 'canceled', 'unavailable', 'invoiced', 'processing', 'created', 'approved'])

### 3.3 Даты — формат и порядок

**Важный момент:** GE проверяет формат строк, **не** конвертированные datetime. Поэтому проверяем по strftime-паттерну исходного CSV. У Olist это `YYYY-MM-DD HH:MM:SS`.

In [8]:
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

# order_purchase_timestamp — обязательная и в правильном формате
orders_suite.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(column='order_purchase_timestamp')
)
orders_suite.add_expectation(
    gxe.ExpectColumnValuesToMatchStrftimeFormat(
        column='order_purchase_timestamp',
        strftime_format=DATE_FORMAT,
    )
)

# Дополнительно: дата покупки в разумных границах (датасет ~2016-2018)
orders_suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(
        column='order_purchase_timestamp',
        min_value='2016-01-01 00:00:00',
        max_value='2019-12-31 23:59:59',
    )
)

ExpectColumnValuesToBeBetween(id='97883e07-1984-4bac-8165-19b69efc75a5', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_purchase_timestamp', mostly=1, row_condition=None, condition_parser=None, min_value=datetime.datetime(2016, 1, 1, 0, 0), max_value=datetime.datetime(2019, 12, 31, 23, 59, 59), strict_min=False, strict_max=False)

### 3.4 Сохраняем suite

В GE 1.x объект suite в памяти и suite в хранилище — это разные вещи. После `add_expectation()` надо вызвать `.save()`, иначе при следующем `get_context()` изменения пропадут.

In [9]:
orders_suite.save()
print(f"Saved {len(orders_suite.expectations)} expectations to {SUITE_NAME_ORDERS}")

Saved 8 expectations to orders_suite


### 3.5 Прогоняем валидацию orders

Сейчас увидим первые результаты. Каждое expectation возвращает success/fail + детали.

In [10]:
validation_def_orders = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="orders_validation",
        data=batch_definitions['orders'],
        suite=orders_suite,
    )
)

result_orders = validation_def_orders.run()

# Краткая сводка
print(f"Success: {result_orders.success}")
print(f"Statistics: {result_orders.statistics}")
print(f"\nFailed expectations:")
for r in result_orders.results:
    if not r.success:
        exp_type = r.expectation_config.type
        kwargs = r.expectation_config.kwargs
        col = kwargs.get('column', '—')
        print(f"  ❌ {exp_type} on '{col}'")

Calculating Metrics:   0%|          | 0/43 [00:00<?, ?it/s]

Success: False
Statistics: {'evaluated_expectations': 8, 'successful_expectations': 7, 'unsuccessful_expectations': 1, 'success_percent': 87.5}

Failed expectations:
  ❌ expect_column_values_to_be_between on 'order_purchase_timestamp'


**Ожидаемо что-то завалится** — в этом весь смысл валидации. Например, `customer_id not null` может пройти, а `order_delivered_customer_date` мы намеренно не проверили на not null, потому что для статуса не-delivered эта колонка легитимно пустая.

Это разделение «технически валидно vs бизнес-валидно» — то, ради чего нужен следующий уровень проверок (custom checks с условиями), который сделаем в день 3.

## 4. TODO: Suite для `order_items`

Из вашего каталога: #8 (`price > 0`), #13 (lead_time — но он в orders).

Минимум проверок:
- `order_id` not null
- `order_item_id` not null
- `product_id` not null
- `seller_id` not null
- Составной PK уникален: `(order_id, order_item_id)`
- `price > 0`
- `freight_value >= 0` (доставка может быть 0 — бесплатная)
- `shipping_limit_date` в формате `%Y-%m-%d %H:%M:%S`

**Подсказки:**
- Составной ключ: `gxe.ExpectCompoundColumnsToBeUnique(column_list=['order_id', 'order_item_id'])`
- Минимум для price: `gxe.ExpectColumnValuesToBeBetween(column='price', min_value=0, strict_min=True)`. `strict_min=True` означает `> 0`, без него было бы `>= 0`.

In [11]:
SUITE_NAME_ITEMS = "order_items_suite"

if SUITE_NAME_ITEMS in [s.name for s in context.suites.all()]:
    context.suites.delete(SUITE_NAME_ITEMS)

items_suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME_ITEMS))

# TODO: добавьте expectations по списку выше
# items_suite.add_expectation(...)
# items_suite.add_expectation(...)

items_suite.save()
print(f"order_items_suite: {len(items_suite.expectations)} expectations")

order_items_suite: 0 expectations


In [12]:
# Набор проверок для состава заказов
items_suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='order_id'))
items_suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='product_id'))

# Составной ключ должен быть уникальным (кейс из Дня 1)
items_suite.add_expectation(
    gxe.ExpectCompoundColumnsToBeUnique(column_list=['order_id', 'order_item_id'])
)

# Проверка цен (из твоего каталога: price > 0.5)
items_suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(column='price', min_value=0.5)
)

# Доставка может быть бесплатной (>= 0), но не отрицательной
items_suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(column='freight_value', min_value=0)
)

ExpectColumnValuesToBeBetween(id='ac4ce845-6e73-429b-9fe1-da2dd3f0d2cd', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='freight_value', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=None, strict_min=False, strict_max=False)

## 5. TODO: Suite для `products`

Из вашего каталога: #3 (category_name nulls), #10 (weight > 0), #16 (габариты > 0).

Минимум:
- `product_id` not null + unique
- `product_category_name` not null (будет фейлиться на 610 строк — это и есть наша цель: зафиксировать факт)
- `product_weight_g > 0`
- `product_length_cm > 0`
- `product_height_cm > 0`
- `product_width_cm > 0`

**Заметка по #3:** правило `not_null` зафейлится на 610 строках, и это **успех валидации в нашем смысле** — мы зафиксировали проблему. В отчёте Data Docs она будет красным цветом, и это правильно. Дальше бизнес решает: заполнять 'unknown', выкидывать, или вернуть в источник на доработку.

In [14]:
SUITE_NAME_PRODUCTS = "products_suite"

if SUITE_NAME_PRODUCTS in [s.name for s in context.suites.all()]:
    context.suites.delete(SUITE_NAME_PRODUCTS)

products_suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME_PRODUCTS))

# Фиксируем уникальность ID
products_suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column='product_id'))

# Эта проверка ЗАФЕЙЛИТСЯ (красный цвет), так как у нас есть null в категориях.
# Это ПРАВИЛЬНО. Мы фиксируем проблему.
products_suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='product_category_name'))

# Проверка веса (кейс из Дня 1: вес должен быть > 0)
products_suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(column='product_weight_g', min_value=0, strict_min=True)
)

products_suite.save()
print(f"products_suite: {len(products_suite.expectations)} expectations")

products_suite: 3 expectations


## 6. TODO: Suite для `order_payments`

Из вашего каталога: #7 (`payment_type` closed set, исключая 'not_defined'), #9 (`payment_value > 0`).

Минимум:
- `order_id` not null
- `payment_sequential > 0`
- Составной PK уникален: `(order_id, payment_sequential)`
- `payment_type` ∈ {credit_card, boleto, voucher, debit_card} — **намеренно исключаем 'not_defined'** чтобы поймать те 3 записи
- `payment_installments >= 1`
- `payment_value > 0`

In [15]:
SUITE_NAME_PAYMENTS = "order_payments_suite"

if SUITE_NAME_PAYMENTS in [s.name for s in context.suites.all()]:
    context.suites.delete(SUITE_NAME_PAYMENTS)

payments_suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME_PAYMENTS))

# Ожидаем только известные типы оплат
payments_suite.add_expectation(
    gxe.ExpectColumnValuesToBeInSet(
        column='payment_type',
        value_set=['credit_card', 'boleto', 'voucher', 'debit_card']
    )
)

# Сумма оплаты должна быть положительной (ловим те 9 нулевых платежей)
payments_suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(column='payment_value', min_value=0, strict_min=True)
)

payments_suite.save()
print(f"order_payments_suite: {len(payments_suite.expectations)} expectations")

order_payments_suite: 2 expectations


## 7. TODO: Suite для `order_reviews`

Из вашего каталога: #5 (814 дублей review_id).

Минимум:
- `review_id` not null
- `review_id` unique (зафейлится на 814 строках — снова, это наша цель: зафиксировать)
- `order_id` not null
- `review_score` ∈ {1, 2, 3, 4, 5}
- `review_creation_date` в формате `%Y-%m-%d %H:%M:%S`

In [23]:
SUITE_NAME_REVIEWS = "order_reviews_suite"

if SUITE_NAME_REVIEWS in [s.name for s in context.suites.all()]:
    context.suites.delete(SUITE_NAME_REVIEWS)

reviews_suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME_REVIEWS))

# Проверка уникальности ID отзыва (тоже ЗАФЕЙЛИТСЯ - это наша цель)
reviews_suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column='review_id'))

# Оценка должна быть строго от 1 до 5
reviews_suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(column='review_score', min_value=1, max_value=5)
)

reviews_suite.save()
print(f"order_reviews_suite: {len(reviews_suite.expectations)} expectations")

order_reviews_suite: 2 expectations


In [24]:
# Добавляем валидации для всех остальных таблиц
# (Убедись, что ты запустила ячейки с TODO 4, 5, 6, 7 до этого!)

context.validation_definitions.add(
    gx.ValidationDefinition(name="order_items_val", data=batch_definitions['order_items'], suite=items_suite)
)
context.validation_definitions.add(
    gx.ValidationDefinition(name="products_val", data=batch_definitions['products'], suite=products_suite)
)
context.validation_definitions.add(
    gx.ValidationDefinition(name="payments_val", data=batch_definitions['order_payments'], suite=payments_suite)
)
context.validation_definitions.add(
    gx.ValidationDefinition(name="reviews_val", data=batch_definitions['order_reviews'], suite=reviews_suite)
)

print("Все валидации успешно зарегистрированы!")

Все валидации успешно зарегистрированы!


## 8. Checkpoint: запускаем всё и генерируем Data Docs

Checkpoint оборачивает все validation definitions и при запуске:
1. Прогоняет все валидации
2. Сохраняет результаты
3. Обновляет Data Docs (HTML-отчёт)

После запуска отчёт автоматически откроется в браузере.

In [25]:
# Собираем все validation definitions, которые есть в проекте
all_validation_defs = list(context.validation_definitions.all())
print(f"Will run {len(all_validation_defs)} validation definitions:")
for vd in all_validation_defs:
    print(f"  - {vd.name}")

Will run 5 validation definitions:
  - orders_validation
  - order_items_val
  - payments_val
  - products_val
  - reviews_val


In [26]:
CHECKPOINT_NAME = "olist_data_audit"

if CHECKPOINT_NAME in [c.name for c in context.checkpoints.all()]:
    context.checkpoints.delete(CHECKPOINT_NAME)

checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name=CHECKPOINT_NAME,
        validation_definitions=all_validation_defs,
        actions=[
            gx.checkpoint.UpdateDataDocsAction(name="update_data_docs"),
        ],
    )
)

checkpoint_result = checkpoint.run()
print(f"\nOverall success: {checkpoint_result.success}")

Calculating Metrics:   0%|          | 0/43 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/33 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/20 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/22 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/17 [00:00<?, ?it/s]


Overall success: False


In [27]:
# Открываем Data Docs в браузере
context.open_data_docs()

**Что вы должны увидеть в браузере:**
- Сводная страница со списком всех проверок
- Для каждой suite — детали: какие expectations прошли (зелёные), какие зафейлились (красные)
- Для зафейленных — счётчик «N unexpected values» и примеры этих значений

**Это и есть тот самый артефакт**, скриншот которого пойдёт в README проекта.

## 9. Что дальше (день 3)

К концу дня 2 у вас:
- ✅ 5 Expectation Suites
- ✅ Checkpoint, который прогоняет всё одной командой
- ✅ Data Docs — HTML-отчёт с реальными находками на 100k+ строк данных

День 3 закрывает три темы:
1. **Cleansing module** (`src/cleanser.py`) — функции исправления типичных проблем (дедуп, type coercion, fillna со стратегиями)
2. **Custom checks** — то, что не выражается стандартными expectations: cross-table FK, conditional expectations (`delivered_date NOT NULL ⟺ status == 'delivered'`)
3. **README** — главная страница проекта с hero-блоком, скриншотами и `Quickstart`